# Wage efficiency in European football — 02 · Model & ranking

**Business question (sporting director view):** given what a club pays its squad, how many points should it get — and how much money is it wasting?

Method, kept deliberately simple and explainable:
1. One linear regression per league and season: `PPG = a + b · log(wage bill)`. Log because each extra million buys fewer points at the top.
2. **Residual** = actual PPG − expected PPG → over/under-performance, converted to points over the season.
3. **Wasted €** = wage bill − the wage bill the model needs for the points actually won. Positive = overpaid, negative = got results cheaply.
4. Quadrants by wage level (above/below league median) × performance (above/below expected).

Runs on the full panel (`club_season_panel.csv`) if it exists, otherwise on the pilot files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('/content/drive/MyDrive/proyecto_eficiencia')
PROCESSED = BASE / 'data' / 'processed'
FIGS = BASE / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
panel_path = PROCESSED / 'club_season_panel.csv'
if panel_path.exists():
    df = pd.read_csv(panel_path)
else:
    df = pd.concat([pd.read_csv(p) for p in sorted(PROCESSED.glob('club_season_*_*.csv'))], ignore_index=True)

df['gd_pg'] = df.gd / df.played
print(df.groupby(['league', 'season']).size().unstack(fill_value=0))
print('\nmissing wages:', df.annual_wages_eur.isna().sum(), '| missing squad value:', df.squad_value_eur.isna().sum())

## 1. Model

One fit per league-season: avoids comparing nominal euros across years (wage inflation) and across leagues (Premier League money).

In [ ]:
def fit_line(x, y):
    # OLS y = a + b*x, returns a, b, r2
    b, a = np.polyfit(x, y, 1)
    pred = a + b * x
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return a, b, r2

def efficiency(df, spend='annual_wages_eur', target='ppg', scope=('league',)):
    # one log-linear model per group; residual = over/under-performance
    out, fits = [], []
    for key, g in df.dropna(subset=[spend, target]).groupby(list(scope)):
        g = g.copy()
        x = np.log(g[spend])
        a, b, r2 = fit_line(x, g[target])
        g['expected'] = a + b * x
        g['residual'] = g[target] - g['expected']
        # spend the model needs for the actual output; capped so title winners don't explode
        needed = np.exp((g[target] - a) / b)
        g['spend_needed'] = np.minimum(needed, 3 * g[spend].max())
        g['wasted_eur'] = g[spend] - g['spend_needed']
        fits.append(dict(zip(scope, key if isinstance(key, tuple) else (key,)), a=a, b=b, r2=r2, n=len(g)))
        out.append(g)
    return pd.concat(out), pd.DataFrame(fits)

def add_quadrants(df, spend='annual_wages_eur', scope=('league', 'season')):
    med = df.groupby(list(scope))[spend].transform('median')
    high = df[spend] >= med
    good = df['residual'] >= 0
    df['quadrant'] = np.select(
        [high & good, high & ~good, ~high & good],
        ['Spends well', 'Loses money', 'Model to copy'], 'Saves but does not compete')
    return df


SCOPE = ('league', 'season')

res, fits = efficiency(df, spend='annual_wages_eur', target='ppg', scope=SCOPE)
res = add_quadrants(res, scope=SCOPE)
res['pts_above_expected'] = res.residual * res.played

fits.round(3)

`b` = extra PPG per log-unit of wages (doubling the wage bill adds `b · 0.69` PPG). `r2` = share of PPG differences explained by wages alone.

## 2. Efficiency ranking

In [ ]:
def fmt(d):
    t = d[['league', 'season', 'team', 'position', 'annual_wages_eur', 'ppg', 'expected', 'pts_above_expected', 'wasted_eur', 'quadrant']].copy()
    t['annual_wages_eur'] = (t.annual_wages_eur / 1e6).round(1)
    t['wasted_eur'] = (t.wasted_eur / 1e6).round(1)
    return t.rename(columns={'annual_wages_eur': 'wages_m', 'wasted_eur': 'wasted_m', 'expected': 'expected_ppg'}).round(2)

ranking = fmt(res.sort_values('pts_above_expected', ascending=False))
ranking.reset_index(drop=True)

## 3. Charts

In [ ]:
def plot_fit(league, season):
    g = res[(res.league == league) & (res.season == season)]
    f = fits[(fits.league == league) & (fits.season == season)].iloc[0]
    colors = {'Spends well': '#2a9d8f', 'Model to copy': '#8ab17d', 'Loses money': '#e76f51', 'Saves but does not compete': '#b0b0b0'}
    fig, ax = plt.subplots(figsize=(10, 6.5))
    for q, gq in g.groupby('quadrant'):
        ax.scatter(gq.annual_wages_eur / 1e6, gq.ppg, s=60, color=colors[q], label=q, zorder=3)
    xs = np.linspace(g.annual_wages_eur.min(), g.annual_wages_eur.max(), 200)
    ax.plot(xs / 1e6, f.a + f.b * np.log(xs), color='#444', lw=1.2, ls='--', label='Expected PPG')
    for _, r in g.iterrows():
        ax.annotate(r.team, (r.annual_wages_eur / 1e6, r.ppg), fontsize=8, xytext=(4, 3), textcoords='offset points')
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    ax.xaxis.set_minor_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}' if str(int(v))[0] in '25' else ''))
    ax.set_xlabel('Annual wage bill (€m, log scale)')
    ax.set_ylabel('Points per game')
    ax.set_title(f'{league} {season}: wages vs points (R² = {f.r2:.2f})')
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGS / f'fit_{league}_{season}.png', bbox_inches='tight')
    plt.show()

for lg, ss in fits[['league', 'season']].itertuples(index=False):
    plot_fit(lg, ss)

In [ ]:
def plot_wasted(d, title, fname, n=8):
    top = d.nlargest(n, 'wasted_eur')
    bottom = d.nsmallest(n, 'wasted_eur')
    t = pd.concat([top, bottom]).drop_duplicates().sort_values('wasted_eur')
    labels = t.team + ' ' + t.season.str[2:4] + '/' + t.season.str[-2:]
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.5))
    ax.barh(labels, t.wasted_eur / 1e6, color=np.where(t.wasted_eur > 0, '#e76f51', '#2a9d8f'))
    ax.axvline(0, color='#444', lw=0.8)
    ax.set_xlabel('Wasted wages (€m)   ← got results cheaply  |  overpaid →')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(FIGS / fname, bbox_inches='tight')
    plt.show()

plot_wasted(res, 'Who overpaid for their results?', 'wasted_all.png')

## 4. Robustness checks

Does the ranking hold if we change the spend metric (squad market value) or the performance metric (goal difference per game, final position)?
Spearman correlation between rankings: close to 1 = same story.

In [ ]:
def rank_corr(other, key=('league', 'season', 'team')):
    a = res.set_index(list(key)).residual
    b = other.set_index(list(key)).residual
    j = pd.concat([a, b], axis=1, join='inner')
    return j.iloc[:, 0].corr(j.iloc[:, 1], method='spearman'), len(j)

def top_overlap(other, n=10):
    a = set(res.nlargest(n, 'residual').team + res.nlargest(n, 'residual').season)
    b = set(other.nlargest(n, 'residual').team + other.nlargest(n, 'residual').season)
    return len(a & b)

variants = {
    'squad value instead of wages': efficiency(df, spend='squad_value_eur', target='ppg', scope=SCOPE)[0],
    'goal difference per game': efficiency(df, spend='annual_wages_eur', target='gd_pg', scope=SCOPE)[0],
    'final position': efficiency(df.assign(neg_position=-df.position), spend='annual_wages_eur', target='neg_position', scope=SCOPE)[0],
}
rows = []
for name, v in variants.items():
    rho, n = rank_corr(v)
    rows.append({'variant': name, 'spearman': round(rho, 2), 'top10_overlap': top_overlap(v), 'n': n})
pd.DataFrame(rows)

## 5. Data quality

Capology estimates part of each wage bill. If over/under-performance correlated with `% estimated`, the ranking could be an artefact of bad wage data.

In [ ]:
q = res.dropna(subset=['pct_estimated'])
print('corr residual vs % estimated:', round(q.residual.corr(q.pct_estimated, method='spearman'), 2))
print('mean % estimated by quadrant:')
q.groupby('quadrant').pct_estimated.mean().round(1)

## 6. Five-season view (full panel)

Cumulative wasted wages and points above expected per club across all seasons it played in the league.

In [ ]:
career = (res.groupby(['league', 'team'])
          .agg(seasons=('season', 'nunique'),
               wages_m=('annual_wages_eur', lambda s: s.sum() / 1e6),
               pts_above_expected=('pts_above_expected', 'sum'),
               wasted_m=('wasted_eur', lambda s: s.sum() / 1e6))
          .round(1).sort_values('wasted_m', ascending=False))
career.head(15)

In [ ]:
res.to_csv(PROCESSED / 'efficiency_results.csv', index=False)
fits.to_csv(PROCESSED / 'model_fits.csv', index=False)
career.reset_index().to_csv(PROCESSED / 'efficiency_career.csv', index=False)
print('saved: efficiency_results.csv, model_fits.csv, efficiency_career.csv  |  figures in', FIGS)

## Limitations to state in the post

- Wages are Capology estimates for most clubs outside the elite (see `% estimated`).
- League games only: European/cup workload is not modelled.
- Wages measure the cost of the squad, not the transfer fees (amortisation) behind it.
- Wasted € is asymmetric: overpayment is bounded by the club's own wage bill, but the "saving" of a title winner grows exponentially
  (the model needs a huge bill for 2.5 PPG). Lead the story with the overpaid side and use points above expected for over-performers.
- Association, not causation: a high residual can mean good management, luck, or an underpriced squad.